# Lab 02 · How Spark plans a query

**Part A** (about 45 min): read query plans, find Catalyst's rewrites, and time Tungsten and an RDD.<br>
**Part B** (about 75 min): 14 problems with `groupBy`, joins, pivot and SQL.

**Checklist questions Part A gives you evidence for:**<br>
<span style="color:#d93025"><b>1.6</b> Catalyst Optimizer · <b>1.8</b> Tungsten · <b>1.13</b> DataFrame vs. RDD</span>

## How to work
* Run the **Setup** cell first, then answer the questions in order. `F` is `pyspark.sql.functions`.
* **Part A**: run the code under the question (or add a line), read the output, and write your answer as a comment.
* **Part B**: write the code yourself. Your result must look exactly like the **expected output**, with the rows in the same order.
* The answers are collapsed at the very end of the notebook. Open one only after you have tried.

## Documentation
* [EXPLAIN](https://spark.apache.org/docs/latest/sql-ref-syntax-qry-explain.html): the four plans and the explain modes
* [Deep dive into the Catalyst optimizer](https://www.databricks.com/blog/2015/04/13/deep-dive-into-spark-sqls-catalyst-optimizer.html): analysis, logical optimization, physical planning, code generation
* [Project Tungsten](https://www.databricks.com/blog/2015/04/28/project-tungsten-bringing-spark-closer-to-bare-metal.html): compact binary rows and generated code
* [Spark SQL guide](https://spark.apache.org/docs/latest/sql-programming-guide.html): DataFrames and how they differ from RDDs
* [Broadcast joins](https://spark.apache.org/docs/latest/sql-performance-tuning.html#automatically-broadcasting-joins): `spark.sql.autoBroadcastJoinThreshold`
* [Leveraging statistics](https://spark.apache.org/docs/latest/sql-performance-tuning.html#leveraging-statistics) and [ANALYZE TABLE](https://spark.apache.org/docs/latest/sql-ref-syntax-aux-analyze-table.html)
* [Grouping](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/grouping.html): `groupBy(...).agg(...)` and `pivot`
* [DataFrame.join](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.DataFrame.join.html): the join types (`inner`, `left`, `left_anti`, ...)
* [Running SQL queries](https://spark.apache.org/docs/latest/sql-getting-started.html#running-sql-queries-programmatically): `spark.sql` on views
* [Functions](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/functions.html): `count`, `sum`, `avg`, `count_distinct`, `when`, `month`, `datediff`

## The data
Three small tables of an online shop. The setup cell creates them, and you can read every row in it.

* **customers**: *customer_id*, *name*, *email*, *city*, *signup_date* (the day the customer registered)
* **products**: *product_id*, *product_name*, *category*, *price* (dollars for one item)
* **orders**: *order_id*, *customer_id*, *product_id*, *quantity*, *order_date*, *status* (`delivered`, `shipped` or `cancelled`)

Good to know:
* order 1011 is a guest checkout: it has no *customer_id*
* customer 6 has no email and no city
* customer 7 has never ordered, and product 107 has never been sold
* order 1016 is for product 108, which is no longer in *products*

These are the same tables as in Lab 01.

## Setup
Run this cell once. It starts Spark, saves the three tables as Parquet files in `data/shop/`, loads them into `customers`, `products` and `orders`, and makes them SQL views with the same names.

In [ ]:
# Setup: starts Spark and creates the three shop tables. Run it once.
import logging, os, shutil, sys, time
from datetime import date
os.environ.setdefault("PYSPARK_PYTHON", sys.executable)   # Spark's Python workers use this notebook's Python
from pyspark.sql import SparkSession, functions as F

spark = (SparkSession.builder
         .master("local[4]")   # 4 tasks at a time on your machine
         .appName("lab02")
         .config("spark.sql.adaptive.enabled", "false")       # AQE stays off until Lab 03, so plans stay simple
         .config("spark.sql.session.timeZone", "UTC")
         .config("spark.sql.shuffle.partitions", "4")          # small data: 4 are enough (Lab 01, A8)
         .config("spark.sql.warehouse.dir", os.path.abspath("data/shop/warehouse"))   # tables of question A10
         .config("spark.ui.showConsoleProgress", "false")
         .config("spark.log.level", "FATAL")                  # hide Spark's log messages
         # Spark 4.2.0 bug (SPARK-60294): Python UDFs on Parquet can hang without this
         .config("spark.sql.execution.arrow.pythonUDF.columnarInput.enabled", "false")
         .getOrCreate())
for name in ("DataFrameQueryContextLogger", "SQLQueryContextLogger"):
    logging.getLogger(name).setLevel(logging.CRITICAL)        # errors still show, without an extra JSON log

customers_rows = [
    # customer_id, name, email, city, signup_date
    (1, "Aziz Karimov",     "aziz@mail.uz",     "Tashkent", date(2024, 1, 5)),
    (2, "Bella Schmidt",    "bella@web.de",     "Berlin",   date(2024, 1, 20)),
    (3, "Carlos Ruiz",      "carlos@correo.es", "Madrid",   date(2024, 2, 2)),
    (4, "Dilnoza Rahimova", "dilnoza@mail.uz",  "Tashkent", date(2024, 2, 14)),
    (5, "Emma Wilson",      "emma@post.co.uk",  "London",   date(2024, 3, 1)),
    (6, "Farid Aliyev",     None,               None,       date(2024, 3, 10)),
    (7, "Grace Lee",        "grace@web.de",     "Berlin",   date(2024, 3, 15)),
]
products_rows = [
    # product_id, product_name, category, price
    (101, "Laptop",       "Electronics", 1200.0),
    (102, "Headphones",   "Electronics",  150.0),
    (103, "Coffee Maker", "Kitchen",       85.0),
    (104, "Blender",      "Kitchen",       60.0),
    (105, "Novel",        "Books",         15.0),
    (106, "Cookbook",     "Books",         25.0),
    (107, "Desk Lamp",    "Home",          40.0),
]
orders_rows = [
    # order_id, customer_id, product_id, quantity, order_date, status
    (1001, 1,    101, 1, date(2024, 1, 10), "delivered"),
    (1002, 1,    105, 3, date(2024, 1, 10), "delivered"),
    (1003, 2,    102, 2, date(2024, 1, 25), "delivered"),
    (1004, 3,    103, 1, date(2024, 2, 3),  "cancelled"),
    (1005, 2,    106, 1, date(2024, 2, 5),  "delivered"),
    (1006, 4,    104, 2, date(2024, 2, 15), "shipped"),
    (1007, 1,    102, 1, date(2024, 2, 20), "delivered"),
    (1008, 5,    101, 1, date(2024, 3, 2),  "shipped"),
    (1009, 3,    105, 4, date(2024, 3, 5),  "delivered"),
    (1010, 4,    106, 2, date(2024, 3, 8),  "cancelled"),
    (1011, None, 104, 1, date(2024, 3, 9),  "delivered"),
    (1012, 6,    103, 2, date(2024, 3, 12), "shipped"),
    (1013, 2,    101, 1, date(2024, 3, 18), "delivered"),
    (1014, 5,    105, 5, date(2024, 3, 20), "delivered"),
    (1015, 1,    103, 1, date(2024, 3, 25), "shipped"),
    (1016, 3,    108, 1, date(2024, 3, 28), "delivered"),
]

# Write the tables as Parquet files and read them back, the way real data arrives.
# data/shop belongs to this lab: it is rebuilt every time you run this cell.
shutil.rmtree("data/shop", ignore_errors=True)
for table, rows, schema in [
    ("customers", customers_rows, "customer_id INT, name STRING, email STRING, city STRING, signup_date DATE"),
    ("products",  products_rows,  "product_id INT, product_name STRING, category STRING, price DOUBLE"),
    ("orders",    orders_rows,    "order_id INT, customer_id INT, product_id INT, quantity INT, order_date DATE, status STRING"),
]:
    spark.createDataFrame(rows, schema).coalesce(1).write.parquet(f"data/shop/{table}")

customers = spark.read.parquet("data/shop/customers")
products = spark.read.parquet("data/shop/products")
orders = spark.read.parquet("data/shop/orders")

# The same tables as SQL views, for spark.sql (question B13)
customers.createOrReplaceTempView("customers")
products.createOrReplaceTempView("products")
orders.createOrReplaceTempView("orders")

print("Spark", spark.version, "| Spark UI:", spark.sparkContext.uiWebUrl)

Look at the three tables:

In [ ]:
customers.show()
products.show()
orders.show()

## Part A · How Spark plans a query

### Catalyst: from your code to a plan <span style="color:#d93025">(1.6)</span>

**A1. The column name below is misspelled. Does Spark report the error right away, or only at the first action?**

In [ ]:
# My answer:
oops = orders.select("quantiti")

**A2. Print all plans of `q` with `q.explain(True)`. What are the four plans called, from top to bottom?**

In [ ]:
# My answer:
q = (orders.join(customers, "customer_id")
           .filter(F.col("city") == "Berlin")
           .select("order_id", "name", "quantity"))

**A3. In the Optimized Logical Plan of `q`, is the `city = Berlin` filter above or below the Join?**

In [ ]:
# My answer:

**A4. Which filters in the Optimized Logical Plan of `q` did you never write?**

In [ ]:
# My answer:

**A5. Find `ReadSchema` in the Physical Plan of `q`. Which columns does Spark read from the orders file?**

In [ ]:
# My answer:

**A6. What does `F.lit(1) + F.lit(1)` turn into in the Optimized Logical Plan?**

In [ ]:
# My answer:
orders.filter(F.col("quantity") > F.lit(1) + F.lit(1)).explain(True)

**A7. Which join algorithm does the Physical Plan of `q` use? Turn broadcast joins off, build `q` again, and look again.**

In [ ]:
# My answer:
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", -1)    # -1 = never broadcast
# ...build q again and explain it, then switch broadcasting back on:
# spark.conf.unset("spark.sql.autoBroadcastJoinThreshold")

### Tungsten <span style="color:#d93025">(1.8)</span>

**A8. A `*(1)` in front of operators means they run together as one generated Java function (whole-stage code generation). How much faster is the query below with it than without it?**

In [ ]:
# My answer:
def big_query():
    return spark.range(200_000_000).filter(F.col("id") % 7 == 0).agg(F.sum("id")).first()[0]

big_query()                                     # warm-up run
for codegen in ["true", "false"]:
    spark.conf.set("spark.sql.codegen.wholeStage", codegen)
    start = time.time()
    big_query()
    print(f"code generation {codegen:5}: {time.time() - start:.2f} s")
spark.conf.set("spark.sql.codegen.wholeStage", "true")

### DataFrame vs. RDD <span style="color:#d93025">(1.13)</span>

**A9. The cell counts the numbers from 0 to 49,999,999 that divide by 7, with an RDD. Count the same with a DataFrame (`spark.range`) and time both. Which is faster, and why?**

In [ ]:
# My answer:
start = time.time()
rdd_count = spark.sparkContext.range(50_000_000).filter(lambda x: x % 7 == 0).count()
print("RDD:      ", rdd_count, f"{time.time() - start:.2f} s")

### Beyond the checklist: table statistics <span style="color:#d93025">(deep dive)</span>

**A10. Does the optimizer know how many rows `orders_t` has? Compare the `Statistics(...)` line before and after `ANALYZE TABLE`.**

In [ ]:
# My answer:
orders.write.mode("overwrite").saveAsTable("orders_t")
spark.conf.set("spark.sql.cbo.enabled", "true")       # let the optimizer use table statistics
spark.table("orders_t").explain("cost")               # before
spark.sql("ANALYZE TABLE orders_t COMPUTE STATISTICS")
spark.table("orders_t").explain("cost")               # after

**Done with Part A?** Explain each red checklist question at the top aloud, in your own words, using what you saw.

## Part B · Aggregations and joins
14 problems on the three tables. Same rule as in Lab 01: you are done when your table looks exactly like the expected output.

**B1. How many orders are there of each status? Sort by status.**

Expected output:

```
+---------+-----+
|   status|count|
+---------+-----+
|cancelled|    2|
|delivered|   10|
|  shipped|    4|
+---------+-----+
```

<details><summary>Hint</summary>

`groupBy("status").count()`

</details>

**B2. For each category, show the number of products, the average price (2 decimals) and the highest price. Sort by category.**

Expected output:

```
+-----------+--------+---------+---------+
|   category|products|avg_price|max_price|
+-----------+--------+---------+---------+
|      Books|       2|     20.0|     25.0|
|Electronics|       2|    675.0|   1200.0|
|       Home|       1|     40.0|     40.0|
|    Kitchen|       2|     72.5|     85.0|
+-----------+--------+---------+---------+
```

<details><summary>Hint</summary>

`groupBy(...).agg(...)` with several functions, each named with `.alias(...)`.

</details>

**B3. How many items of each product were delivered? Show product_id and items, most items first; on a tie, the smaller product_id first.**

Expected output:

```
+----------+-----+
|product_id|items|
+----------+-----+
|       105|   12|
|       102|    3|
|       101|    2|
|       104|    1|
|       106|    1|
|       108|    1|
+----------+-----+
```

<details><summary>Hint</summary>

Filter first, then `F.sum("quantity")`.

</details>

**B4. Which customers placed 3 or more orders? Show customer_id and orders, sorted by customer_id.**

Expected output:

```
+-----------+------+
|customer_id|orders|
+-----------+------+
|          1|     4|
|          2|     3|
|          3|     3|
+-----------+------+
```

<details><summary>Hint</summary>

Aggregate, then `filter` on the new column (this is SQL's `HAVING`).

</details>

**B5. How many different customers ordered in each month? Show month (1 to 3) and customers.**

Expected output:

```
+-----+---------+
|month|customers|
+-----+---------+
|    1|        2|
|    2|        4|
|    3|        6|
+-----+---------+
```

<details><summary>Hint</summary>

`F.month(...)`, `F.count_distinct(...)`.

</details>

**B6. Show the orders of customers who live in Berlin: order_id, name and order_date, sorted by order_id.**

Expected output:

```
+--------+-------------+----------+
|order_id|         name|order_date|
+--------+-------------+----------+
|    1003|Bella Schmidt|2024-01-25|
|    1005|Bella Schmidt|2024-02-05|
|    1013|Bella Schmidt|2024-03-18|
+--------+-------------+----------+
```

<details><summary>Hint</summary>

`orders.join(customers, "customer_id")` joins on the column both tables have.

</details>

**B7. For every delivered order, show order_id, product_name, quantity, price and revenue (quantity × price). Highest revenue first; on a tie, the smaller order_id first.**

Expected output:

```
+--------+------------+--------+------+-------+
|order_id|product_name|quantity| price|revenue|
+--------+------------+--------+------+-------+
|    1001|      Laptop|       1|1200.0| 1200.0|
|    1013|      Laptop|       1|1200.0| 1200.0|
|    1003|  Headphones|       2| 150.0|  300.0|
|    1007|  Headphones|       1| 150.0|  150.0|
|    1014|       Novel|       5|  15.0|   75.0|
|    1009|       Novel|       4|  15.0|   60.0|
|    1011|     Blender|       1|  60.0|   60.0|
|    1002|       Novel|       3|  15.0|   45.0|
|    1005|    Cookbook|       1|  25.0|   25.0|
+--------+------------+--------+------+-------+
```

<details><summary>Hint</summary>

Join orders with products, then compute the new column.

</details>

**B8. What is the revenue of delivered orders in each category? Show category and revenue, highest first.**

Expected output:

```
+-----------+-------+
|   category|revenue|
+-----------+-------+
|Electronics| 2850.0|
|      Books|  205.0|
|    Kitchen|   60.0|
+-----------+-------+
```

<details><summary>Hint</summary>

Join first, then group. `F.sum` can add up an expression, not only a column.

</details>

**B9. Which customers have never placed an order? Show customer_id and name.**

Expected output:

```
+-----------+---------+
|customer_id|     name|
+-----------+---------+
|          7|Grace Lee|
+-----------+---------+
```

<details><summary>Hint</summary>

The join type `"left_anti"` keeps the rows of the left table that have no match.

</details>

**B10. Show every customer with their number of orders, including customers with 0 orders. Show name and orders, most orders first; on a tie, by name.**

Expected output:

```
+----------------+------+
|            name|orders|
+----------------+------+
|    Aziz Karimov|     4|
|   Bella Schmidt|     3|
|     Carlos Ruiz|     3|
|Dilnoza Rahimova|     2|
|     Emma Wilson|     2|
|    Farid Aliyev|     1|
|       Grace Lee|     0|
+----------------+------+
```

<details><summary>Hint</summary>

A `left` join keeps customers without orders. Then count a column that is empty (null) for them.

</details>

**B11. For each customer who placed orders, count their delivered and their cancelled orders in two columns. Show customer_id, delivered and cancelled, sorted by customer_id. Leave out the guest order.**

Expected output:

```
+-----------+---------+---------+
|customer_id|delivered|cancelled|
+-----------+---------+---------+
|          1|        3|        0|
|          2|        3|        0|
|          3|        2|        1|
|          4|        0|        1|
|          5|        1|        0|
|          6|        0|        0|
+-----------+---------+---------+
```

<details><summary>Hint</summary>

`F.sum(F.when(condition, 1).otherwise(0))` counts the rows where the condition is true.

</details>

**B12. Make a table with one row per month and one column per status, holding the number of orders, with 0 where there are none. Columns: month, cancelled, delivered, shipped.**

Expected output:

```
+-----+---------+---------+-------+
|month|cancelled|delivered|shipped|
+-----+---------+---------+-------+
|    1|        0|        3|      0|
|    2|        1|        2|      1|
|    3|        1|        5|      3|
+-----+---------+---------+-------+
```

<details><summary>Hint</summary>

`groupBy(...).pivot("status", [...]).count()` turns values into columns; `na.fill(0)` replaces the nulls.

</details>

**B13. With SQL (`spark.sql`): how much did each customer spend on delivered orders? Show name and spent (the sum of quantity × price), highest first. The tables are also SQL views called customers, products and orders.**

Expected output:

```
+-------------+------+
|         name| spent|
+-------------+------+
|Bella Schmidt|1525.0|
| Aziz Karimov|1395.0|
|  Emma Wilson|  75.0|
|  Carlos Ruiz|  60.0|
+-------------+------+
```

<details><summary>Hint</summary>

`spark.sql("""SELECT ... FROM orders o JOIN customers c ON ... JOIN products p ON ... WHERE ... GROUP BY ... ORDER BY ...""")`

</details>

**B14. How many days after registering did each customer place their first order? Show name, signup_date, first_order and days_to_first_order, sorted by signup_date.**

Expected output:

```
+----------------+-----------+-----------+-------------------+
|            name|signup_date|first_order|days_to_first_order|
+----------------+-----------+-----------+-------------------+
|    Aziz Karimov| 2024-01-05| 2024-01-10|                  5|
|   Bella Schmidt| 2024-01-20| 2024-01-25|                  5|
|     Carlos Ruiz| 2024-02-02| 2024-02-03|                  1|
|Dilnoza Rahimova| 2024-02-14| 2024-02-15|                  1|
|     Emma Wilson| 2024-03-01| 2024-03-02|                  1|
|    Farid Aliyev| 2024-03-10| 2024-03-12|                  2|
+----------------+-----------+-----------+-------------------+
```

<details><summary>Hint</summary>

Two steps: the first order date per customer (`F.min("order_date")`), then join it with customers and use `F.datediff`.

</details>

---
**Done?** In the Theory part of Day 2, answer the red checklist questions aloud.

## Answers
Try each question yourself first. Click a question to open its answer; Part A answers also show the output of the reference run.

<details>
<summary><b>A1</b> &nbsp;The column name below is misspelled. Does Spark report the error right away, or only at the first action?</summary>

**Right away**, before any action. Building a DataFrame runs Catalyst's first phase, **analysis**: it looks up every table and column name and works out the types. A wrong name fails here. Wrong data, like the division by zero in Lab 01, fails only when a job runs.

```python
oops = orders.select("quantiti")
```

Result:

```
AnalysisException: [UNRESOLVED_COLUMN.WITH_SUGGESTION] A column, variable, or function parameter with name `quantiti` cannot be resolved. Did you mean one of the following? [`quantity`, `status`, `order_id`, `product_id`, `customer_id`]. SQLSTATE: 42703;
```

</details>

<details>
<summary><b>A2</b> &nbsp;Print all plans of q with q.explain(True). What are the four plans called, from top to bottom?</summary>

**Parsed Logical Plan** (your code as a tree; names Spark has not resolved yet start with `'`), **Analyzed Logical Plan** (names and types resolved), **Optimized Logical Plan** (after Catalyst's rewrite rules) and **Physical Plan** (the operators that really run, with the join algorithm and the file scans). These are Catalyst's phases: analysis → logical optimization → physical planning. Code generation then turns the physical plan into Java code (question A8).

```python
q = (orders.join(customers, "customer_id")
           .filter(F.col("city") == "Berlin")
           .select("order_id", "name", "quantity"))
q.explain(True)
```

Result:

```
== Parsed Logical Plan ==
'Project ['order_id, 'name, 'quantity]
+- Filter (city#18 = Berlin)
   +- Project [customer_id#25, order_id#24, product_id#26, quantity#27, order_date#28, status#29, name#16, email#17, city#18, signup_date#19]
      +- Join Inner, (customer_id#25 = customer_id#15)
         :- Relation [order_id#24,customer_id#25,product_id#26,quantity#27,order_date#28,status#29] parquet
         +- Relation [customer_id#15,name#16,email#17,city#18,signup_date#19] parquet

== Analyzed Logical Plan ==
order_id: int, name: string, quantity: int
Project [order_id#24, name#16, quantity#27]
+- Filter (city#18 = Berlin)
   +- Project [customer_id#25, order_id#24, product_id#26, quantity#27, order_date#28, status#29, name#16, email#17, city#18, signup_date#19]
      +- Join Inner, (customer_id#25 = customer_id#15)
         :- Relation [order_id#24,customer_id#25,product_id#26,quantity#27,order_date#28,status#29] parquet
         +- Relation [customer_id#15,name#16,email#17,city#18,signup_date#19] parquet

== Optimized Logical Plan ==
Project [order_id#24, name#16, quantity#27]
+- Join Inner, (customer_id#25 = customer_id#15)
   :- Project [order_id#24, customer_id#25, quantity#27]
   :  +- Filter isnotnull(customer_id#25)
   :     +- Relation [order_id#24,customer_id#25,product_id#26,quantity#27,order_date#28,status#29] parquet
   +- Project [customer_id#15, name#16]
      +- Filter ((isnotnull(city#18) AND (city#18 = Berlin)) AND isnotnull(customer_id#15))
         +- Relation [customer_id#15,name#16,email#17,city#18,signup_date#19] parquet

== Physical Plan ==
*(2) Project [order_id#24, name#16, quantity#27]
+- *(2) BroadcastHashJoin [customer_id#25], [customer_id#15], Inner, BuildRight, false, false
   :- *(2) Filter isnotnull(customer_id#25)
   :  +- *(2) ColumnarToRow
   :     +- FileScan parquet [order_id#24,customer_id#25,quantity#27] Batched: true, DataFilters: [isnotnull(customer_id#25)], Format: Parquet, Location: InMemoryFileIndex(1 paths)[file:/..., PartitionFilters: [], PushedFilters: [IsNotNull(customer_id)], ReadSchema: struct<order_id:int,customer_id:int,quantity:int>
   +- BroadcastExchange HashedRelationBroadcastMode(List(cast(input[0, int, true] as bigint)),false), [plan_id=161]
      +- *(1) Project [customer_id#15, name#16]
         +- *(1) Filter ((isnotnull(city#18) AND (city#18 = Berlin)) AND isnotnull(customer_id#15))
            +- *(1) ColumnarToRow
               +- FileScan parquet [customer_id#15,name#16,city#18] Batched: true, DataFilters: [isnotnull(city#18), (city#18 = Berlin), isnotnull(customer_id#15)], Format: Parquet, Location: InMemoryFileIndex(1 paths)[file:/..., PartitionFilters: [], PushedFilters: [IsNotNull(city), EqualTo(city,Berlin), IsNotNull(customer_id)], ReadSchema: struct<customer_id:int,name:string,city:string>
```

</details>

<details>
<summary><b>A3</b> &nbsp;In the Optimized Logical Plan of q, is the city = Berlin filter above or below the Join?</summary>

**Below the Join, on the customers side.** You wrote the filter after the join, but Catalyst moved it before the join (**predicate pushdown**), so only Berlin customers go into the join.

Result:

```
== Optimized Logical Plan ==
Project [order_id#24, name#16, quantity#27]
+- Join Inner, (customer_id#25 = customer_id#15)
   :- Project [order_id#24, customer_id#25, quantity#27]
   :  +- Filter isnotnull(customer_id#25)
   :     +- Relation [order_id#24,customer_id#25,product_id#26,quantity#27,order_date#28,status#29] parquet
   +- Project [customer_id#15, name#16]
      +- Filter ((isnotnull(city#18) AND (city#18 = Berlin)) AND isnotnull(customer_id#15))
         +- Relation [customer_id#15,name#16,email#17,city#18,signup_date#19] parquet
```

</details>

<details>
<summary><b>A4</b> &nbsp;Which filters in the Optimized Logical Plan of q did you never write?</summary>

**`isnotnull(customer_id)` on both sides, and `isnotnull(city)`.** A row with a null key never matches in an inner join, and `city = 'Berlin'` is never true for a null city, so Catalyst adds these filters (rule *InferFiltersFromConstraints*) to drop such rows as early as possible.

Result:

```
   :  +- Filter isnotnull(customer_id#25)
      +- Filter ((isnotnull(city#18) AND (city#18 = Berlin)) AND isnotnull(customer_id#15))
```

</details>

<details>
<summary><b>A5</b> &nbsp;Find ReadSchema in the Physical Plan of q. Which columns does Spark read from the orders file?</summary>

**Only `order_id`, `customer_id` and `quantity`.** The query needs no other column, so the others are never read (**column pruning**). The filters are pushed into the file reader too: see `PushedFilters` of the customers scan, which lets Parquet skip data that cannot match.

Result:

```
   :     +- FileScan parquet [order_id#24,customer_id#25,quantity#27] Batched: true, DataFilters: [isnotnull(customer_id#25)], Format: Parquet, Location: InMemoryFileIndex(1 paths)[file:/..., PartitionFilters: [], PushedFilters: [IsNotNull(customer_id)], ReadSchema: struct<order_id:int,customer_id:int,quantity:int>
               +- FileScan parquet [customer_id#15,name#16,city#18] Batched: true, DataFilters: [isnotnull(city#18), (city#18 = Berlin), isnotnull(customer_id#15)], Format: Parquet, Location: InMemoryFileIndex(1 paths)[file:/..., PartitionFilters: [], PushedFilters: [IsNotNull(city), EqualTo(city,Berlin), IsNotNull(customer_id)], ReadSchema: struct<customer_id:int,name:string,city:string>
```

</details>

<details>
<summary><b>A6</b> &nbsp;What does F.lit(1) + F.lit(1) turn into in the Optimized Logical Plan?</summary>

**`quantity > 2`.** Catalyst computes constant expressions once while planning (rule **ConstantFolding**) instead of once per row. (With plain Python, `F.col("quantity") > 1 + 1`, Python adds the numbers before Spark ever sees them.)

```python
orders.filter(F.col("quantity") > F.lit(1) + F.lit(1)).explain(True)
```

Result:

```
== Parsed Logical Plan ==
'Filter '`>`('quantity, '`+`(1, 1))
+- Relation [order_id#24,customer_id#25,product_id#26,quantity#27,order_date#28,status#29] parquet

== Analyzed Logical Plan ==
order_id: int, customer_id: int, product_id: int, quantity: int, order_date: date, status: string
Filter (quantity#27 > (1 + 1))
+- Relation [order_id#24,customer_id#25,product_id#26,quantity#27,order_date#28,status#29] parquet

== Optimized Logical Plan ==
Filter (isnotnull(quantity#27) AND (quantity#27 > 2))
+- Relation [order_id#24,customer_id#25,product_id#26,quantity#27,order_date#28,status#29] parquet

== Physical Plan ==
*(1) Filter (isnotnull(quantity#27) AND (quantity#27 > 2))
+- *(1) ColumnarToRow
   +- FileScan parquet [order_id#24,customer_id#25,product_id#26,quantity#27,order_date#28,status#29] Batched: true, DataFilters: [isnotnull(quantity#27), (quantity#27 > 2)], Format: Parquet, Location: InMemoryFileIndex(1 paths)[file:/..., PartitionFilters: [], PushedFilters: [IsNotNull(quantity), GreaterThan(quantity,2)], ReadSchema: struct<order_id:int,customer_id:int,product_id:int,quantity:int,order_date:date,status:string>
```

</details>

<details>
<summary><b>A7</b> &nbsp;Which join algorithm does the Physical Plan of q use? Turn broadcast joins off, build q again, and look again.</summary>

**`BroadcastHashJoin`, then `SortMergeJoin`.** customers is tiny, under the 10 MB `spark.sql.autoBroadcastJoinThreshold`, so Spark sends a copy of it to every executor and joins without shuffling orders. With broadcasting off, both sides are shuffled (two `Exchange`s) and sorted by customer_id before they are merged. Choosing the algorithm is **physical planning**. Lab 05 goes deeper into joins.

```python
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", -1)    # -1 = never broadcast
q = (orders.join(customers, "customer_id")
           .filter(F.col("city") == "Berlin")
           .select("order_id", "name", "quantity"))
q.explain()
spark.conf.unset("spark.sql.autoBroadcastJoinThreshold")   # back to the default, 10 MB
```

Result:

```
== Physical Plan ==
*(5) Project [order_id#24, name#16, quantity#27]
+- *(5) SortMergeJoin [customer_id#25], [customer_id#15], Inner
   :- *(2) Sort [customer_id#25 ASC NULLS FIRST], false, 0
   :  +- Exchange hashpartitioning(customer_id#25, 4), ENSURE_REQUIREMENTS, [plan_id=235]
   :     +- *(1) Filter isnotnull(customer_id#25)
   :        +- *(1) ColumnarToRow
   :           +- FileScan parquet [order_id#24,customer_id#25,quantity#27] Batched: true, DataFilters: [isnotnull(customer_id#25)], Format: Parquet, Location: InMemoryFileIndex(1 paths)[file:/..., PartitionFilters: [], PushedFilters: [IsNotNull(customer_id)], ReadSchema: struct<order_id:int,customer_id:int,quantity:int>
   +- *(4) Sort [customer_id#15 ASC NULLS FIRST], false, 0
      +- Exchange hashpartitioning(customer_id#15, 4), ENSURE_REQUIREMENTS, [plan_id=245]
         +- *(3) Project [customer_id#15, name#16]
            +- *(3) Filter ((isnotnull(city#18) AND (city#18 = Berlin)) AND isnotnull(customer_id#15))
               +- *(3) ColumnarToRow
                  +- FileScan parquet [customer_id#15,name#16,city#18] Batched: true, DataFilters: [isnotnull(city#18), (city#18 = Berlin), isnotnull(customer_id#15)], Format: Parquet, Location: InMemoryFileIndex(1 paths)[file:/..., PartitionFilters: [], PushedFilters: [IsNotNull(city), EqualTo(city,Berlin), IsNotNull(customer_id)], ReadSchema: struct<customer_id:int,name:string,city:string>
```

</details>

<details>
<summary><b>A8</b> &nbsp;A *(1) in front of operators means they run together as one generated Java function (whole-stage code ...</summary>

**Several times faster**: compare the two times in the result. Without code generation, every operator hands rows to the next one through function calls, row by row. With it, Spark writes one tight Java loop for the whole stage (filter and sum together) and keeps rows in its compact binary format instead of Java objects. Both are **Tungsten**. `explain("codegen")` shows the generated code.

```python
def big_query():
    return spark.range(200_000_000).filter(F.col("id") % 7 == 0).agg(F.sum("id")).first()[0]

big_query()                                     # warm-up run
for codegen in ["true", "false"]:
    spark.conf.set("spark.sql.codegen.wholeStage", codegen)
    start = time.time()
    big_query()
    print(f"code generation {codegen:5}: {time.time() - start:.2f} s")
spark.conf.set("spark.sql.codegen.wholeStage", "true")
```

Result:

```
code generation true : 0.11 s
code generation false: 0.68 s
```

*Your times will differ.*

</details>

<details>
<summary><b>A9</b> &nbsp;The cell counts the numbers from 0 to 49,999,999 that divide by 7, with an RDD. Count the same with a ...</summary>

**The DataFrame, by far**: compare the two times in the result. The RDD's `lambda` is a black box: Spark must send every number to a Python process, run your function there and get the answer back. The DataFrame's `F.col("id") % 7 == 0` is an expression Spark understands: Catalyst can optimize it and Tungsten runs it as generated Java code. A DataFrame has a schema (named, typed columns); an RDD is a collection of objects Spark cannot look into.

```python
start = time.time()
rdd_count = spark.sparkContext.range(50_000_000).filter(lambda x: x % 7 == 0).count()
print("RDD:      ", rdd_count, f"{time.time() - start:.2f} s")

start = time.time()
df_count = spark.range(50_000_000).filter(F.col("id") % 7 == 0).count()
print("DataFrame:", df_count, f"{time.time() - start:.2f} s")
```

Result:

```
RDD:       7142858 1.21 s
DataFrame: 7142858 0.08 s
```

*Your times will differ.*

</details>

<details>
<summary><b>A10</b> &nbsp;Does the optimizer know how many rows orders_t has? Compare the Statistics(...) line before and after ANALYZE ...</summary>

**Not before, yes after.** Before, the plan knows only `sizeInBytes` (the file size). After `ANALYZE TABLE`, it also knows `rowCount=16`, and estimates the size from it. The optimizer uses statistics like these to estimate how big every step is, for example to decide whether a table is small enough to broadcast, or in which order to join several tables (cost-based optimization).

```python
orders.write.mode("overwrite").saveAsTable("orders_t")
spark.conf.set("spark.sql.cbo.enabled", "true")       # let the optimizer use table statistics
spark.table("orders_t").explain("cost")               # before
spark.sql("ANALYZE TABLE orders_t COMPUTE STATISTICS")
spark.table("orders_t").explain("cost")               # after
```

Result:

```
Relation spark_catalog.default.orders_t[order_id#120,customer_id#121,product_id#122,quantity#123,order_date#124,status#125] parquet, Statistics(sizeInBytes=2.0 KiB)
Relation spark_catalog.default.orders_t[order_id#142,customer_id#143,product_id#144,quantity#145,order_date#146,status#147] parquet, Statistics(sizeInBytes=768.0 B, rowCount=16)
```

</details>

<details>
<summary><b>B1</b> &nbsp;How many orders are there of each status? Sort by status.</summary>

```python
orders.groupBy("status").count().orderBy("status").show()
```

</details>

<details>
<summary><b>B2</b> &nbsp;For each category, show the number of products, the average price (2 decimals) and the highest price. Sort by ...</summary>

```python
(products.groupBy("category")
         .agg(F.count("*").alias("products"),
              F.round(F.avg("price"), 2).alias("avg_price"),
              F.max("price").alias("max_price"))
         .orderBy("category")
         .show())
```

</details>

<details>
<summary><b>B3</b> &nbsp;How many items of each product were delivered? Show product_id and items, most items first; on a tie, the ...</summary>

```python
(orders.filter(F.col("status") == "delivered")
       .groupBy("product_id")
       .agg(F.sum("quantity").alias("items"))
       .orderBy(F.col("items").desc(), "product_id")
       .show())
```

</details>

<details>
<summary><b>B4</b> &nbsp;Which customers placed 3 or more orders? Show customer_id and orders, sorted by customer_id.</summary>

```python
(orders.groupBy("customer_id")
       .agg(F.count("*").alias("orders"))
       .filter(F.col("orders") >= 3)
       .orderBy("customer_id")
       .show())
```

</details>

<details>
<summary><b>B5</b> &nbsp;How many different customers ordered in each month? Show month (1 to 3) and customers.</summary>

```python
(orders.groupBy(F.month("order_date").alias("month"))
       .agg(F.count_distinct("customer_id").alias("customers"))
       .orderBy("month")
       .show())
```

`count_distinct` skips nulls, so the guest order (no customer_id) does not count in March.

</details>

<details>
<summary><b>B6</b> &nbsp;Show the orders of customers who live in Berlin: order_id, name and order_date, sorted by order_id.</summary>

```python
(orders.join(customers, "customer_id")
       .filter(F.col("city") == "Berlin")
       .select("order_id", "name", "order_date")
       .orderBy("order_id")
       .show())
```

</details>

<details>
<summary><b>B7</b> &nbsp;For every delivered order, show order_id, product_name, quantity, price and revenue (quantity × price). ...</summary>

```python
(orders.filter(F.col("status") == "delivered")
       .join(products, "product_id")
       .withColumn("revenue", F.col("quantity") * F.col("price"))
       .select("order_id", "product_name", "quantity", "price", "revenue")
       .orderBy(F.col("revenue").desc(), "order_id")
       .show())
```

Order 1016 is missing from the result: its product 108 is not in *products*, and an inner join keeps only rows that match on both sides.

</details>

<details>
<summary><b>B8</b> &nbsp;What is the revenue of delivered orders in each category? Show category and revenue, highest first.</summary>

```python
(orders.filter(F.col("status") == "delivered")
       .join(products, "product_id")
       .groupBy("category")
       .agg(F.sum(F.col("quantity") * F.col("price")).alias("revenue"))
       .orderBy(F.col("revenue").desc())
       .show())
```

</details>

<details>
<summary><b>B9</b> &nbsp;Which customers have never placed an order? Show customer_id and name.</summary>

```python
customers.join(orders, "customer_id", "left_anti").select("customer_id", "name").show()
```

</details>

<details>
<summary><b>B10</b> &nbsp;Show every customer with their number of orders, including customers with 0 orders. Show name and orders, ...</summary>

```python
(customers.join(orders, "customer_id", "left")
          .groupBy("customer_id", "name")
          .agg(F.count("order_id").alias("orders"))
          .select("name", "orders")
          .orderBy(F.col("orders").desc(), "name")
          .show())
```

`F.count("order_id")` skips nulls, so Grace gets 0. `F.count("*")` would give her 1: her joined row exists, it just has empty order columns.

</details>

<details>
<summary><b>B11</b> &nbsp;For each customer who placed orders, count their delivered and their cancelled orders in two columns. Show ...</summary>

```python
(orders.filter(F.col("customer_id").isNotNull())
       .groupBy("customer_id")
       .agg(F.sum(F.when(F.col("status") == "delivered", 1).otherwise(0)).alias("delivered"),
            F.sum(F.when(F.col("status") == "cancelled", 1).otherwise(0)).alias("cancelled"))
       .orderBy("customer_id")
       .show())
```

</details>

<details>
<summary><b>B12</b> &nbsp;Make a table with one row per month and one column per status, holding the number of orders, with 0 where ...</summary>

```python
(orders.groupBy(F.month("order_date").alias("month"))
       .pivot("status", ["cancelled", "delivered", "shipped"])
       .count()
       .na.fill(0)
       .orderBy("month")
       .show())
```

</details>

<details>
<summary><b>B13</b> &nbsp;With SQL (spark.sql): how much did each customer spend on delivered orders? Show name and spent (the sum of ...</summary>

```python
spark.sql("""
    SELECT c.name, SUM(o.quantity * p.price) AS spent
    FROM orders o
    JOIN customers c ON o.customer_id = c.customer_id
    JOIN products p ON o.product_id = p.product_id
    WHERE o.status = 'delivered'
    GROUP BY c.name
    ORDER BY spent DESC
""").show()
```

</details>

<details>
<summary><b>B14</b> &nbsp;How many days after registering did each customer place their first order? Show name, signup_date, ...</summary>

```python
first_orders = orders.groupBy("customer_id").agg(F.min("order_date").alias("first_order"))
(customers.join(first_orders, "customer_id")
          .withColumn("days_to_first_order", F.datediff("first_order", "signup_date"))
          .select("name", "signup_date", "first_order", "days_to_first_order")
          .orderBy("signup_date")
          .show())
```

Grace has no orders, so the inner join leaves her out.

</details>